# 06 - Modelo Avanzado (Fine-Tuning Transformer)

**Track:** Tarea de Investigación
**Responsable:** Sebastián Ruiz

Este notebook implementa el fine-tuning de un modelo pre-entrenado (ej. BETO) respetando el contrato del equipo:
- Semilla fija (42)
- Carga de splits generados por Yibby (`../data/train.csv`)
- Exportación estandarizada de métricas en JSON (`../results/transformer_metrics.json`)


In [8]:
import torch
import numpy as np
import random
import os
from pathlib import Path

# ── CONTRATO 4.3: SEMILLA FIJA = 42 ─────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
elif torch.backends.mps.is_available():
    torch.mps.manual_seed(SEED)

# ── DEVICE: compatible con CUDA (Colab/GPU), MPS (Mac M2) y CPU ──────────────
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
else:
    DEVICE = torch.device('cpu')

print(f'Semilla fija : {SEED}')
print(f'Dispositivo  : {DEVICE}')
print(f'PyTorch      : {torch.__version__}')

# ── CONTRATO 4.1: RUTAS RELATIVAS ────────────────────────────────────────────
DATA_DIR    = Path('../data')
RESULTS_DIR = Path('../results')
FIGURES_DIR = Path('../figures')

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_PATH = RESULTS_DIR / 'transformer_best.pt'
print(f'DATA_DIR    : {DATA_DIR}')
print(f'RESULTS_DIR : {RESULTS_DIR}')
print(f'FIGURES_DIR : {FIGURES_DIR}')


Semilla fija : 42
Dispositivo  : mps
PyTorch      : 2.12.0
DATA_DIR    : ../data
RESULTS_DIR : ../results
FIGURES_DIR : ../figures


## Instalación de dependencias

Ejecutar solo si el entorno no tiene los paquetes instalados.


In [ ]:
# Descomentar y ejecutar si faltan dependencias
# !pip install transformers accelerate datasets scikit-learn seaborn

## Carga de los splits

Se cargan directamente `train.csv`, `val.csv` y `test.csv` generados desde `src/preprocessing.py`.


In [9]:
import pandas as pd

train_df = pd.read_csv(DATA_DIR / 'train.csv')
val_df   = pd.read_csv(DATA_DIR / 'val.csv')
test_df  = pd.read_csv(DATA_DIR / 'test.csv')

print(f'Train : {len(train_df):>5} muestras')
print(f'Val   : {len(val_df):>5} muestras')
print(f'Test  : {len(test_df):>5} muestras')
print(f'Total : {len(train_df)+len(val_df)+len(test_df):>5} muestras')
print('Distribución train:', train_df['label'].value_counts().sort_index().to_dict())


Train : 12720 muestras
Val   :  2726 muestras
Test  :  2726 muestras
Total : 18172 muestras
Distribución train: {0: 1174, 1: 696, 2: 1592, 3: 2955, 4: 6303}


In [10]:
from transformers import AutoTokenizer

MODEL_NAME = "dccuchile/bert-base-spanish-wwm-cased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Verificar con un ejemplo real del split de entrenamiento
ejemplo = train_df['review_text'].iloc[0]
tokens  = tokenizer(ejemplo, truncation=True, max_length=128, padding='max_length')

print(f"Modelo        : {MODEL_NAME}")
print(f"Vocab size    : {tokenizer.vocab_size}")
print(f"Input IDs     : {tokens['input_ids'][:10]} ...")
print(f"Attention mask: {tokens['attention_mask'][:10]} ...")


Modelo        : dccuchile/bert-base-spanish-wwm-cased
Vocab size    : 31002
Input IDs     : [4, 1177, 1049, 5225, 1311, 6794, 1042, 16232, 1017, 1108] ...
Attention mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1] ...


In [11]:
from torch.utils.data import Dataset, DataLoader

MAX_LEN    = 128   # BETO acepta hasta 512, pero 128 cubre bien reseñas cortas y es más rápido
BATCH_SIZE = 32

class HotelReviewDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts     = texts.reset_index(drop=True)
        self.labels    = labels.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_len,
            padding='max_length',
            return_tensors='pt'
        )
        return {
            'input_ids'     : encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label'         : torch.tensor(self.labels[idx], dtype=torch.long)
        }

# Crear datasets
train_dataset = HotelReviewDataset(train_df['review_text'], train_df['label'], tokenizer, MAX_LEN)
val_dataset   = HotelReviewDataset(val_df['review_text'],   val_df['label'],   tokenizer, MAX_LEN)
test_dataset  = HotelReviewDataset(test_df['review_text'],  test_df['label'],  tokenizer, MAX_LEN)

# Crear DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f"MAX_LEN       : {MAX_LEN}")
print(f"Batch size    : {BATCH_SIZE}")
print(f"Batches train : {len(train_loader)}")
print(f"Batches val   : {len(val_loader)}")
print(f"Batches test  : {len(test_loader)}")


MAX_LEN       : 128
Batch size    : 32
Batches train : 398
Batches val   : 86
Batches test  : 86


In [12]:
from transformers import AutoModelForSequenceClassification

NUM_CLASSES = 5

# Cargar BETO con cabeza de clasificación
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_CLASSES
)

# ── Fine-Tuning: congelar todo excepto el último bloque del encoder + clasificador ──
# BETO tiene 12 capas (encoder.layer.0 ... encoder.layer.11)
# Congelamos todas menos la última (layer.11)

for name, param in model.named_parameters():
    param.requires_grad = False  # Congelar todo primero

for name, param in model.named_parameters():
    if any(layer in name for layer in ['encoder.layer.11', 'pooler', 'classifier']):
        param.requires_grad = True  # Descongelar última capa + clasificador

# Verificar qué quedó descongelado
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Parámetros totales     : {total_params:,}")
print(f"Parámetros entrenables : {trainable_params:,}")
print(f"Porcentaje entrenable  : {100*trainable_params/total_params:.1f}%")
print("\nCapas descongeladas:")
for name, param in model.named_parameters():
    if param.requires_grad:
        print(f"  ✓ {name}")

model = model.to(DEVICE)


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
bert.pooler.dense.bias                     | MISSING    | 
classifier.bias                            | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	

Parámetros totales     : 109,854,725
Parámetros entrenables : 7,682,309
Porcentaje entrenable  : 7.0%

Capas descongeladas:
  ✓ bert.encoder.layer.11.attention.self.query.weight
  ✓ bert.encoder.layer.11.attention.self.query.bias
  ✓ bert.encoder.layer.11.attention.self.key.weight
  ✓ bert.encoder.layer.11.attention.self.key.bias
  ✓ bert.encoder.layer.11.attention.self.value.weight
  ✓ bert.encoder.layer.11.attention.self.value.bias
  ✓ bert.encoder.layer.11.attention.output.dense.weight
  ✓ bert.encoder.layer.11.attention.output.dense.bias
  ✓ bert.encoder.layer.11.attention.output.LayerNorm.weight
  ✓ bert.encoder.layer.11.attention.output.LayerNorm.bias
  ✓ bert.encoder.layer.11.intermediate.dense.weight
  ✓ bert.encoder.layer.11.intermediate.dense.bias
  ✓ bert.encoder.layer.11.output.dense.weight
  ✓ bert.encoder.layer.11.output.dense.bias
  ✓ bert.encoder.layer.11.output.LayerNorm.weight
  ✓ bert.encoder.layer.11.output.LayerNorm.bias
  ✓ bert.pooler.dense.weight
  ✓ bert.pooler

/Users/juanse/Desktop/Proyecto_Deep_Learning/.venv/lib/python3.13/site-packages/huggingface_hub/file_download.py:731: UserWarning: Not enough free disk space to download the file. The expected file size is: 439.56 MB. The target location /Users/juanse/.cache/huggingface/hub/models--dccuchile--bert-base-spanish-wwm-cased/blobs only has 103.02 MB free disk space.
  warnings.warn(


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/threading.py", line 1041, in _bootstrap_inner
    self.run()
    ~~~~~~~~^^
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/threading.py", line 992, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/juanse/Desktop/Proyecto_Deep_Learning/.venv/lib/python3.13/site-packages/transformers/safetensors_conversion.py", line 117, in auto_conversion
    raise e
  File "/Users/juanse/Desktop/Proyecto_Deep_Learning/.venv/lib/python3.13/site-packages/transformers/safetensors_conversion.py", line 113, in auto_conversion
    resolved_archive_file = cached_file(pretrained_model_name_or_path, filename, **cached_file_kwargs)
  File "/Users/juanse/Desktop/Proyecto_Deep_Learning/.venv/lib/python3.13/site-packages/transformers/utils/hub.py", line 278, in cached_file
 

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
from transformers import get_linear_schedule_with_warmup

# Class weights para manejar el desbalance
classes           = np.array([0, 1, 2, 3, 4])
class_weights     = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=train_df['label'].values
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(DEVICE)

print("Class weights:")
for i, w in enumerate(class_weights):
    print(f"  Clase {i+1} (rating {i+1}★): {w:.4f}")

# Función de pérdida con class weights
criterion = torch.nn.CrossEntropyLoss(weight=class_weights_tensor)

# Optimizador — torch.optim.AdamW (AdamW fue removido de transformers)
LR       = 2e-5
N_EPOCHS = 1

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR,
    weight_decay=0.01
)

# Scheduler con warmup
total_steps  = len(train_loader) * N_EPOCHS
warmup_steps = total_steps // 10

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

print(f"\nLearning rate : {LR}")
print(f"Épocas        : {N_EPOCHS}")
print(f"Total steps   : {total_steps}")
print(f"Warmup steps  : {warmup_steps}")


In [ ]:
import time
from tqdm import tqdm

# CHECKPOINT_PATH ya definido en la celda de configuración

def train_epoch(model, loader, optimizer, scheduler, criterion, device):
    model.train()
    total_loss, correct, total = 0, 0, 0

    for batch in tqdm(loader, desc="  Train", leave=False):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['label'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss    = criterion(outputs.logits, labels)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        preds       = outputs.logits.argmax(dim=1)
        correct    += (preds == labels).sum().item()
        total      += labels.size(0)

    return total_loss / len(loader), correct / total


def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0

    with torch.no_grad():
        for batch in tqdm(loader, desc="  Val  ", leave=False):
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['label'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss    = criterion(outputs.logits, labels)

            total_loss += loss.item()
            preds       = outputs.logits.argmax(dim=1)
            correct    += (preds == labels).sum().item()
            total      += labels.size(0)

    return total_loss / len(loader), correct / total


# ── Historial para las curvas ──────────────────────────────────────────────
loss_history     = []
val_loss_history = []
acc_history      = []
val_acc_history  = []
best_val_loss    = float('inf')
best_epoch       = 1
start_time       = time.time()

print(f"{'Época':>6} {'Train Loss':>11} {'Val Loss':>10} {'Train Acc':>10} {'Val Acc':>9} {'★':>3}")
print("─" * 58)

for epoch in range(1, N_EPOCHS + 1):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, scheduler, criterion, DEVICE)
    val_loss,   val_acc   = eval_epoch(model, val_loader, criterion, DEVICE)

    loss_history.append(train_loss)
    val_loss_history.append(val_loss)
    acc_history.append(train_acc)
    val_acc_history.append(val_acc)

    # Guardar mejor modelo
    improved = ""
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch    = epoch
        torch.save(model.state_dict(), CHECKPOINT_PATH)
        improved = "✓"

    print(f"{epoch:>6} {train_loss:>11.4f} {val_loss:>10.4f} {train_acc:>10.4f} {val_acc:>9.4f} {improved:>3}")

elapsed = time.time() - start_time
print(f"\nEntrenamiento completado en {elapsed:.0f}s")
print(f"Mejor época : {best_epoch}  (val_loss={best_val_loss:.4f})")
print(f"Checkpoint  : {CHECKPOINT_PATH}")


In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, f1_score, confusion_matrix

def evaluate_model(model, loader, checkpoint_path, device):
    model.load_state_dict(torch.load(checkpoint_path, map_location=device, weights_only=True))
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch in tqdm(loader, desc="  Test"):
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['label'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds   = outputs.logits.argmax(dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return all_labels, all_preds

y_true, y_pred = evaluate_model(model, test_loader, CHECKPOINT_PATH, DEVICE)

# ── Calcular métricas ──────────────────────────────────────────────────────
accuracy                    = accuracy_score(y_true, y_pred)
precision, recall, f1, _    = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
f1_per_class_arr            = f1_score(y_true, y_pred, average=None, zero_division=0)
cm                          = confusion_matrix(y_true, y_pred).tolist()

CLASS_NAMES  = ["1", "2", "3", "4", "5"]
f1_per_class = {CLASS_NAMES[i]: round(float(v), 6) for i, v in enumerate(f1_per_class_arr)}

print("── Métricas finales (test set) ─────────────────────────")
print(f"  Accuracy        : {accuracy:.4f}")
print(f"  Precision macro : {precision:.4f}")
print(f"  Recall macro    : {recall:.4f}")
print(f"  F1 macro        : {f1:.4f}")
print("  F1 por clase:")
for cls, val in f1_per_class.items():
    print(f"    Clase {cls}★ : {val:.4f}")
print("────────────────────────────────────────────────────────")


In [ ]:
import json

metrics_dict = {
    "model_name": "beto_finetuned",
    "owner": "Sebastián Ruiz",
    "track": "TI",
    "config": {
        "base_model": MODEL_NAME,
        "layers_unfrozen": ["encoder.layer.11", "pooler", "classifier"],
        "max_len": MAX_LEN,
        "batch_size": BATCH_SIZE,
        "lr": LR,
        "weight_decay": 0.01,
        "use_class_weights": True,
        "n_params": sum(p.numel() for p in model.parameters()),
        "trainable_params": sum(p.numel() for p in model.parameters() if p.requires_grad)
    },
    "metrics": {
        "accuracy":         round(float(accuracy), 6),
        "precision_macro":  round(float(precision), 6),
        "recall_macro":     round(float(recall), 6),
        "f1_macro":         round(float(f1), 6),
        "f1_per_class":     f1_per_class,
        "confusion_matrix": cm
    },
    "training": {
        "epochs_run":        N_EPOCHS,
        "best_epoch":        best_epoch,
        "training_time_seconds": int(elapsed),
        "loss_history":      loss_history,
        "val_loss_history":  val_loss_history,
        "acc_history":       acc_history,
        "val_acc_history":   val_acc_history
    }
}

json_path = RESULTS_DIR / "transformer_metrics.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(metrics_dict, f, indent=2, ensure_ascii=False)

print(f"JSON guardado → {json_path}")


In [ ]:
import matplotlib.pyplot as plt

epochs = range(1, N_EPOCHS + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("Curvas de entrenamiento — BETO Fine-tuning", fontsize=13)

# Loss
ax = axes[0]
ax.plot(epochs, loss_history,     label="Train loss", color="#2196F3")
ax.plot(epochs, val_loss_history, label="Val loss",   color="#F44336", linestyle="--")
ax.axvline(best_epoch, color="gray", linestyle=":", alpha=0.8, label=f"Mejor época ({best_epoch})")
ax.set_xlabel("Época"); ax.set_ylabel("Loss")
ax.set_title("Loss (CrossEntropy)"); ax.legend(); ax.grid(True, alpha=0.3)

# Accuracy
ax = axes[1]
ax.plot(epochs, acc_history,     label="Train acc", color="#4CAF50")
ax.plot(epochs, val_acc_history, label="Val acc",   color="#FF9800", linestyle="--")
ax.axvline(best_epoch, color="gray", linestyle=":", alpha=0.8, label=f"Mejor época ({best_epoch})")
ax.set_xlabel("Época"); ax.set_ylabel("Accuracy")
ax.set_title("Accuracy"); ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
curves_path = FIGURES_DIR / "transformer_curves.png"
plt.savefig(curves_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Guardado → {curves_path}")


In [ ]:
import seaborn as sns
import numpy as np

cm_array = np.array(cm)
cm_norm  = cm_array / cm_array.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(
    cm_norm, annot=True, fmt=".2f", cmap="Blues",
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
    ax=ax, linewidths=0.5, linecolor="lightgray"
)
ax.set_title("Matriz de confusión — BETO Fine-tuning\n(normalizada por fila)", fontsize=12)
ax.set_ylabel("Etiqueta real")
ax.set_xlabel("Etiqueta predicha")
plt.tight_layout()

cm_path = FIGURES_DIR / "transformer_confusion.png"
plt.savefig(cm_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Guardado → {cm_path}")
